In [ ]:
import csv
import json
import sys
from pathlib import Path
from collections import Counter
import pandas as pd

BASE = Path("..")

RAW_USERNAMES = BASE / "complete_dataset_raw/raw_actions_compressed.jsonl"
RAW_NUMBERS = BASE / "complete_dataset_raw/raw_numbers_compressed.jsonl"

CSV_NUMBERS = BASE / "dataset_categories/phone_numbers_completo_categories.csv"

OUT = BASE / "cleaning"
OUT.mkdir(parents=True, exist_ok=True)

CLEAN_USERNAMES = BASE / "raw_usernames_compressed.jsonl"

print("RAW usernames:", RAW_USERNAMES.exists())
print("RAW numbers:", RAW_NUMBERS.exists())
print("CSV numbers:", CSV_NUMBERS.exists())
print("Output:", CLEAN_USERNAMES.resolve())

RAW usernames: True
RAW numbers: True
CSV numbers: True
Output: /home/local/ADUNIPI/a.belliani/CNA/raw_usernames_compressed.jsonl


In [2]:
max_size = sys.maxsize

while True:
    try:
        csv.field_size_limit(max_size)
        break
    except OverflowError:
        max_size //= 10

print("CSV field size limit:", csv.field_size_limit())

CSV field size limit: 9223372036854775807


In [3]:
phone_addresses = set()

with open(CSV_NUMBERS, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    if "address" not in reader.fieldnames:
        raise ValueError("Colonna address assente nel CSV dei numbers")

    for row in reader:
        address = (row.get("address") or "").strip()

        if address:
            phone_addresses.add(address)

print(f"Phone numbers distinti nel CSV: {len(phone_addresses):,}")

Phone numbers distinti nel CSV: 136,566


In [4]:
def nome_mancante(value):
    return (
        value is None
        or not isinstance(value, str)
        or value.strip().lower() in {"", "nan", "none", "null"}
    )

In [5]:
def analizza_jsonl(path, layer, filtra_numbers=False):
    tipi = Counter()
    esiti = Counter()
    nft_distinti = set()
    nft_senza_nome = {}
    esempi_tipi = {}

    righe_totali = 0
    righe_non_valide = 0
    righe_numbers_estranei = 0
    nft_numbers_estranei = set()

    output_file = (
        open(CLEAN_USERNAMES, "w", encoding="utf-8")
        if filtra_numbers else None
    )

    try:
        with open(path, "r", encoding="utf-8") as f:
            for line_number, line in enumerate(f, 1):
                if not line.strip():
                    continue

                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    righe_non_valide += 1
                    continue

                nft = r.get("n", r.get("nft"))
                nome = r.get("u", r.get("username"))
                tipo = r.get("y", r.get("action_type", "UNKNOWN"))
                success = r.get("ok", r.get("success"))

                righe_totali += 1

                # Esclude i phone numbers dal RAW usernames
                if filtra_numbers and nft in phone_addresses:
                    righe_numbers_estranei += 1
                    nft_numbers_estranei.add(nft)
                    continue

                if nft:
                    nft_distinti.add(nft)

                    if nome_mancante(nome):
                        nft_senza_nome[nft] = nome

                tipi[tipo] += 1
                esiti[str(success)] += 1

                # Salva un esempio per ogni action_type
                if tipo not in esempi_tipi:
                    esempi_tipi[tipo] = {
                        "action_type": tipo,
                        "nft": nft,
                        "details": json.dumps(
                            r.get("det", r.get("details")),
                            ensure_ascii=False
                        )[:3000]
                    }

                if output_file:
                    output_file.write(line)

    finally:
        if output_file:
            output_file.close()

    # Distribuzione degli action_type
    df_tipi = pd.DataFrame(
        tipi.items(),
        columns=["action_type", "n_actions"]
    ).sort_values("n_actions", ascending=False)

    df_tipi.to_csv(
        OUT / f"action_types_{layer}.csv",
        index=False
    )

    # Esempi details per action_type
    pd.DataFrame(esempi_tipi.values()).to_csv(
        OUT / f"examples_{layer}.csv",
        index=False
    )

    # NFT senza nome
    pd.DataFrame(
        [
            {"nft": n, "asset_name": v}
            for n, v in nft_senza_nome.items()
        ],
        columns=["nft", "asset_name"]
    ).to_csv(
        OUT / f"nft_senza_nome_{layer}.csv",
        index=False
    )

    # Phone numbers erroneamente presenti negli usernames
    if filtra_numbers:
        pd.DataFrame(
            {"nft": sorted(nft_numbers_estranei)}
        ).to_csv(
            OUT / "numbers_nel_raw_usernames.csv",
            index=False
        )

    print(f"\n{'=' * 55}")
    print(f"LAYER: {layer}")
    print(f"{'=' * 55}")

    print(f"Actions lette: {righe_totali:,}")
    print(f"Righe JSON non valide: {righe_non_valide:,}")
    print(f"NFT distinti mantenuti: {len(nft_distinti):,}")
    print(f"NFT senza nome: {len(nft_senza_nome):,}")

    print(f"Actions con success=True: {esiti['True']:,}")
    print(f"Actions con success=False: {esiti['False']:,}")
    print(f"Actions senza esito: {esiti['None']:,}")

    if filtra_numbers:
        print(f"Actions di phone numbers escluse: {righe_numbers_estranei:,}")
        print(f"Phone numbers distinti esclusi: {len(nft_numbers_estranei):,}")

    display(df_tipi)

    return df_tipi, nft_distinti

In [6]:
tipi_usernames, nft_usernames = analizza_jsonl(
    RAW_USERNAMES,
    "usernames",
    filtra_numbers=True
)

print("\nFile usernames pulito:")
print(CLEAN_USERNAMES.resolve())
print("Dimensione (MB):", round(CLEAN_USERNAMES.stat().st_size / 1024**2, 2))


LAYER: usernames
Actions lette: 41,478,299
Righe JSON non valide: 0
NFT distinti mantenuti: 555,207
NFT senza nome: 2
Actions con success=True: 40,390,461
Actions con success=False: 1,051,747
Actions senza esito: 0
Actions di phone numbers escluse: 36,091
Phone numbers distinti esclusi: 480


,action_type,n_actions
3,teleitem_start_auction,17178687
4,unknown,16443826
2,nft_transfer,2538980
6,auction_bid,1610473
8,call_contract,1138105
7,auction_outbid,563171
0,contract_deploy,555207
1,nft_mint,555207
5,teleitem_cancel_auction,376725
10,nft_put_on_sale,142047



File usernames pulito:
/home/local/ADUNIPI/a.belliani/CNA/raw_usernames_compressed.jsonl
Dimensione (MB): 33143.46


In [7]:
tipi_numbers, nft_numbers = analizza_jsonl(
    RAW_NUMBERS,
    "phone_numbers",
    filtra_numbers=False
)


LAYER: phone_numbers
Actions lette: 12,089,699
Righe JSON non valide: 0
NFT distinti mantenuti: 136,566
NFT senza nome: 0
Actions con success=True: 10,778,390
Actions con success=False: 1,311,309
Actions senza esito: 0


,action_type,n_actions
4,teleitem_start_auction,4048276
5,unknown,3673016
7,auction_bid,1379701
3,call_contract,1278565
2,nft_transfer,814356
8,auction_outbid,225031
6,teleitem_cancel_auction,169122
1,nft_mint,136566
0,contract_deploy,136566
9,nft_put_on_sale,84694


In [8]:
overlap = nft_usernames & nft_numbers

print("=" * 55)
print("RIEPILOGO")
print("=" * 55)

print(f"NFT usernames dopo pulizia: {len(nft_usernames):,}")
print(f"NFT phone numbers: {len(nft_numbers):,}")
print(f"NFT presenti in entrambi: {len(overlap):,}")

print("\nFile di output:", OUT.resolve())

if overlap:
    overlap_path = OUT / "overlap_residuo.csv"

    pd.DataFrame({
        "nft": sorted(overlap)
    }).to_csv(overlap_path, index=False)

    print("Overlap residuo salvato in:", overlap_path)
else:
    print("Nessun overlap residuo rilevato.")

RIEPILOGO
NFT usernames dopo pulizia: 555,207
NFT phone numbers: 136,566
NFT presenti in entrambi: 0

File di output: /home/local/ADUNIPI/a.belliani/CNA/cleaning
Nessun overlap residuo rilevato.
